# Terrain, drainage et itinéraires

**Cartomize · Version 1.0**

Durée indicative : 30 minutes.

## Objectifs

Calculer des dérivées topographiques, un drainage D8 et un itinéraire sur un réseau préparé.

## Apport de Cartomize

Les résultats topographiques et les réseaux peuvent être intégrés aux mêmes plans de traitement et mises en page.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("06_terrain_reseaux")
d = demo(out/"data")

## 1. Dérivées du terrain

Le MNT est simulé en mètres, avec des pixels de 30 m. Le TPI, le TRI et la rugosité décrivent des propriétés différentes du voisinage.

In [3]:
terrain=cm.terrain(d['dem'],out/'terrain.tif',products=('slope','aspect','hillshade','tpi','tri','roughness'),block_size=64)
smooth=cm.convolve(d['dem'],out/'mnt_lisse.tif',np.ones((3,3)).tolist(),normalize=True,block_size=64)
with rasterio.open(terrain) as src:
    display(pd.DataFrame({'bande':src.indexes,'produit':src.descriptions}))
fig=show_raster(terrain,'Pente sur MNT simulé',band=1,cmap='magma')
gallery(fig,'06_terrain');plt.show()

,bande,produit
0,1,slope
1,2,aspect
2,3,hillshade
3,4,tpi
4,5,tri
5,6,roughness


## 2. Drainage D8

Priority-Flood comble les dépressions et D8 affecte une direction d’écoulement. L’accumulation compte les cellules contributives, cellule courante comprise. Le seuil du réseau est exprimé en nombre de cellules, pas en débit.

In [4]:
hydro=cm.hydrology(d['dem'],out/'hydrologie',stream_threshold=100,memory_limit_mb=256)
display(hydro)
assert (out/'hydrologie/accumulation.tif').is_file()

PosixPath('tutorials/outputs/06_terrain_reseaux_ww2pl18e/hydrologie/accumulation.tif')

## 3. Plus court chemin

Le réseau pédagogique est bidirectionnel et connecté aux intersections. Les rattachements se font aux nœuds les plus proches. Les distances d’approche ne sont pas incluses dans la longueur sur le réseau.

In [5]:
route=cm.shortest_path(d['roads'],(300300,9498500),(301900,9499800),points_crs=CRS,metric_crs=CRS,max_snap_m=50)
route.to_file(out/'itineraire.gpkg',index=False)
display(route.drop(columns='geometry'))
assert np.isclose(cm.length(route).sum(),2900)

,distance_m,start_snap_m,end_snap_m,nodes
0,2900.0,0.0,0.0,3


## Exercice

Que devient le réseau extrait si le seuil passe de 100 à 300 cellules ? Peut-on interpréter ce résultat comme une mesure de débit ?

<details>
<summary>Éléments de correction</summary>

Le réseau retient moins de cellules. D8 fournit une topologie et une accumulation géométrique ; le débit nécessite notamment des précipitations, des pertes et un modèle hydrologique adapté.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.